In [1]:
import os
from pathlib import Path
from openai import OpenAI
from langchain_openai import ChatOpenAI

In [2]:
from ontovis.vis_agent import *

Loading config from: /home/pascalgrosset/projects/OntoVis/configs/config.yaml


In [3]:
llm = ChatOpenAI(
    model="gpt-5.1",
    api_key=os.environ["OPENAI_API_KEY"]
)

In [4]:
current_path = Path().resolve()
current_path

PosixPath('/home/pascalgrosset/projects/OntoVis')

In [5]:
vis_ai = VisExplorer(llm)

In [6]:
vis_ai.chat("Can you use the vtk package and volume render the image at: /home/pascalgrosset/projects/OntoVis/notebooks/skull_256x256x256_uint8.raw and show me the code and then run it")


-----volume_rendering_instructions---



Python REPL can execute arbitrary code. Use with caution.



-----python_repl_tool---



Here is the full Python script that uses `vtk` to volume render your RAW skull dataset and save a PNG snapshot, followed by confirmation that it has been run.

```python
import vtk
import numpy as np

# Parameters
file_path = '/home/pascalgrosset/projects/OntoVis/notebooks/skull_256x256x256_uint8.raw'
dimensions = (256, 256, 256)  # (nx, ny, nz)
output_image_path = 'skull_volume_render.png'

# 1. Read the raw volume data (uint8)
data = np.fromfile(file_path, dtype=np.uint8)
if data.size != np.prod(dimensions):
    raise ValueError(f'Unexpected data size: {data.size}, expected {np.prod(dimensions)}')

# 2. Convert numpy array to VTK image data
image_data = vtk.vtkImageData()
image_data.SetDimensions(*dimensions)
image_data.AllocateScalars(vtk.VTK_UNSIGNED_CHAR, 1)

# Copy numpy data into VTK image data
vtk_array = image_data.GetPointData().GetScalars()
for i in range(data.size):
    vtk_array.SetTuple1(i, int(data[i]))

# 3. Set up volume rendering
volume_mapper = vtk.vtkSmartVolumeMapper()
volume_mapper.SetInputData(image_data)

# 4. Define opacity (scalar → alpha)
opacity_tf = vtk.vtkPiecewiseFunction()
opacity_tf.AddPoint(0,   0.0)
opacity_tf.AddPoint(50,  0.0)
opacity_tf.AddPoint(80,  0.1)
opacity_tf.AddPoint(120, 0.3)
opacity_tf.AddPoint(255, 0.8)

# 5. Define color (scalar → RGB)
color_tf = vtk.vtkColorTransferFunction()
color_tf.AddRGBPoint(0,   0.0, 0.0, 0.0)
color_tf.AddRGBPoint(80,  0.8, 0.8, 0.8)
color_tf.AddRGBPoint(120, 1.0, 1.0, 1.0)
color_tf.AddRGBPoint(255, 1.0, 1.0, 1.0)

# 6. Volume properties
volume_property = vtk.vtkVolumeProperty()
volume_property.SetColor(color_tf)
volume_property.SetScalarOpacity(opacity_tf)
volume_property.SetInterpolationTypeToLinear()
volume_property.ShadeOn()
volume_property.SetAmbient(0.3)
volume_property.SetDiffuse(0.6)
volume_property.SetSpecular(0.5)

# 7. Volume actor
volume = vtk.vtkVolume()
volume.SetMapper(volume_mapper)
volume.SetProperty(volume_property)

# 8. Renderer, render window
renderer = vtk.vtkRenderer()
renderer.AddVolume(volume)
renderer.SetBackground(0.0, 0.0, 0.0)

render_window = vtk.vtkRenderWindow()
render_window.AddRenderer(renderer)
render_window.SetSize(800, 800)

# Position camera
renderer.ResetCamera()

# 9. Capture the render to an image
window_to_image_filter = vtk.vtkWindowToImageFilter()
window_to_image_filter.SetInput(render_window)
window_to_image_filter.Update()

# 10. Save to PNG
png_writer = vtk.vtkPNGWriter()
png_writer.SetFileName(output_image_path)
png_writer.SetInputConnection(window_to_image_filter.GetOutputPort())
png_writer.Write()

print(f'Volume rendering saved to {output_image_path}')
```

I have executed this script in the current environment; it completed successfully and wrote:

`Volume rendering saved to skull_volume_render.png`

You should now find `skull_volume_render.png` in the working directory where you run this script. If you’d like, I can help you adjust the transfer functions (opacity/color) to better highlight specific structures in the skull.


Query took: 22.61 seconds, total tokens used: 2995
 
